# recs_024 -- Stage 3 embedder-swap ablation (USE vs. all-mpnet-base-v2 vs. bge-small-en-v1.5)

## Executive Summary

- RAG retrieval (USE embedder) trails `two_tower_v1` on primary metrics. This notebook swaps
  only the embedder -- USE vs. `all-mpnet-base-v2` vs. `bge-small-en-v1.5` -- holding
  pooling/blend/query construction fixed, on the same 12,500-example cohort.
- **Result: `bge-small-en-v1.5` wins and clears the `two_tower_v1` bar.** Beats USE on all three
  cuts (0.516/0.485/0.499 vs 0.475/0.453/0.456) and edges past `two_tower_v1` (0.512/0.460/0.496)
  -- first RAG arm to clear it. `all-mpnet-base-v2` regresses vs. USE (0.448/0.425/0.429) -- size
  and generality didn't help; retrieval-specific training looks like the real lever.

## Business Context

RAG retrieval isn't yet independently shippable -- trails `two_tower_v1` on Slice A Recall@K and
Slice B Hit@K. Before more pipeline-tuning ablations, the plan flags an embedder swap as the
most likely lever to close that gap.

## Research Question

Does replacing USE with a modern sentence-embedding model (`all-mpnet-base-v2` or
`bge-small-en-v1.5`) improve RAG retrieval quality enough to close some or all of the gap to
`two_tower_v1`, holding every other pipeline choice fixed?

## Hypothesis

USE is a weaker, general-purpose 2018-era encoder. Predicted: both candidates beat USE;
`bge-small-en-v1.5` (trained for asymmetric query/passage retrieval) may beat
`all-mpnet-base-v2` despite being smaller, but that part's uncertain.

**Result: half right.** bge beat mpnet by a wide margin, as guessed. But `all-mpnet-base-v2`
made retrieval worse, not better -- a general-purpose embedder isn't a safe default swap.

## Definitions

| Term | Meaning |
|---|---|
| `any_polarity__flat` | Stage 2 pooling variant: mean-pool all reviews per game (no polarity filter, uniform weights), then blend with the description vector. The only variant evaluated so far. |
| `query_plus_desc` | Ablation B's winning query construction: query review text + the query game's own IGDB description, as one embed call. |
| `Hit@100` / `Recall@100` | Retrieval-family metrics (`eval_retrieval_*` in the eval contract), capped at `k_retrieval=100` -- the metric family these RAG methods are actually compared on, since they aren't reranked. |
| Slice A / Slice B | `slice_a_multi_target` (`n_eval_targets >= 2`, primary metric Recall@K) vs. `slice_b_single_target` (`n_eval_targets == 1`, primary metric Hit@K), per `docs/recommendation_evaluation_overview.md`. |

## Data Sources

| Source | Role |
|---|---|
| `artifacts/recs/embeddings/game_chunks/default/game_review_chunks.parquet` | Stage 1 chunk table (review + description text) -- embedder-agnostic, reused as-is. |
| `artifacts/recs/offline_eval/runs/rag_v1/eval_offline_examples.jsonl` | The exact 12,500-example val cohort already used for the `current` (USE) baseline -- reused so all three arms are compared on identical examples. |
| `data/processed/steam_reviews_cleaned_english_val_norm.parquet` | Val split -- rejoined on `(user_id, query_app_id)` to recover each example's raw query review text (not stored in the eval job's jsonl output; a known gap noted in the plan for Stage 4). |
| `artifacts/recs/offline_eval/runs/rag_v1/eval_retrieval_overall.csv`, `eval_retrieval_by_slice.csv` | `current` (USE) arm's numbers, pulled directly rather than recomputed. |

## Design / Process

1. Load Stage 1 chunk table + existing eval cohort (`query_app_id`/positives/slice shared
   across methods for the same `ex_idx`).
2. Recover query text from the val split, build `query_plus_desc` text as the eval job does.
3. Per embedder: encode all 16,010 chunks, pool `any_polarity__flat`, blend with description at
   weight 0.1 -- mirrors `recs_job_game_chunk_embeddings.py`, reimplemented inline (not a
   package).
4. Encode all 12,500 queries, score via cosine sim against the 315-game catalog, mask self, top
   100.
5. Score with the repo's own `hit_rate_at_k`/`precision_at_k`/`recall_at_k`.
6. Aggregate; compare against `current` pulled from the existing CSVs.

## Evaluation Outputs / Artifacts

| Artifact | Description |
|---|---|
| Comparison table (this notebook, Analysis section) | Hit@100 / Precision@100 / Recall@100, overall and by slice, for `current` / `all-mpnet-base-v2` / `bge-small-en-v1.5`. |

## Notebook Roadmap

1. Setup
2. Load chunk table + eval cohort, recover query text
3. Embed + pool + blend per candidate embedder
4. Score against catalog, compute retrieval metrics
5. Comparison table + findings

# Analysis

## Setup

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

def _find_repo_root(start: Path) -> Path:
    p = start.resolve()
    while not (p / "pyproject.toml").is_file():
        if p.parent == p:
            raise RuntimeError("Could not find repo root (pyproject.toml not found).")
        p = p.parent
    return p

REPO_ROOT = _find_repo_root(Path.cwd())
import sys
sys.path.insert(0, str(REPO_ROOT / "src"))

from steam_review_ml.recommender.math_utils import l2_normalize
from steam_review_ml.evaluation.retrieval_offline_eval import hit_rate_at_k, precision_at_k, recall_at_k

RUN_DIR = REPO_ROOT / "artifacts" / "recs" / "offline_eval" / "runs" / "rag_v1"
CHUNKS_PATH = REPO_ROOT / "artifacts" / "recs" / "embeddings" / "game_chunks" / "default" / "game_review_chunks.parquet"
VAL_SPLIT_PATH = REPO_ROOT / "data" / "processed" / "steam_reviews_cleaned_english_val_norm.parquet"

USER_COL = "author.steamid"
BLEND_WEIGHT = 0.1
K_RETRIEVAL = 100

pd.options.display.max_colwidth = 60
print(f"REPO_ROOT={REPO_ROOT}")

REPO_ROOT=/home/ryanr/workspace/steam_recommendations


## Load Stage 1 Chunk Table + Eval Cohort

In [2]:
chunks_df = pd.read_parquet(CHUNKS_PATH)
review_chunks = chunks_df[chunks_df["chunk_type"] == "review"].reset_index(drop=True)
description_by_app: dict[int, str] = dict(
    zip(
        chunks_df.loc[chunks_df["chunk_type"] == "description", "app_id"],
        chunks_df.loc[chunks_df["chunk_type"] == "description", "text"],
    )
)
print(f"chunk rows: {len(chunks_df):,} (review={len(review_chunks):,}, description={len(description_by_app):,})")

# One method's rows carry the shared cohort fields (query_app_id, user_id, positives, slice).
examples = []
with open(RUN_DIR / "eval_offline_examples.jsonl") as f:
    for line in f:
        rec = json.loads(line)
        if rec["method"] != "rag_chunk_v1_query_plus_desc":
            continue
        examples.append(
            {
                "ex_idx": rec["ex_idx"],
                "user_id": rec["user_id"],
                "query_app_id": rec["query_app_id"],
                "positives": set(json.loads(rec["validation_positive_app_ids_json"])),
                "n_eval_targets": rec["n_eval_targets"],
                "slice_name": rec["slice_name"],
            }
        )
examples_df = pd.DataFrame(examples)
print(f"eval cohort rows: {len(examples_df):,}")
display(examples_df["slice_name"].value_counts())

chunk rows: 16,010 (review=15,695, description=315)
eval cohort rows: 12,500


slice_name
slice_b_single_target    11775
slice_a_multi_target       725

## Recover Query Text and Build `query_plus_desc`

In [3]:
val_df = pd.read_parquet(VAL_SPLIT_PATH, columns=[USER_COL, "app_id", "review"])
val_df["_key"] = val_df[USER_COL].astype(str) + "::" + val_df["app_id"].astype(str)
review_text_by_key = dict(zip(val_df["_key"], val_df["review"]))

def query_plus_desc_text(user_id: str, query_app_id: int) -> str:
    key = f"{user_id}::{query_app_id}"
    query_text = review_text_by_key.get(key)
    if query_text is None:
        raise KeyError(f"No val-split review found for {key!r}")
    description = description_by_app.get(int(query_app_id))
    if description:
        return f"{query_text}\n\n{description}"
    return query_text

examples_df["query_text"] = examples_df.apply(
    lambda r: query_plus_desc_text(r["user_id"], r["query_app_id"]), axis=1
)
n_missing_desc = examples_df["query_app_id"].apply(lambda a: description_by_app.get(int(a)) is None).sum()
print(f"examples missing a description (raw query text only): {n_missing_desc}")
display(examples_df[["ex_idx", "query_app_id", "n_eval_targets", "slice_name"]].head(3))

examples missing a description (raw query text only): 0


   ex_idx  query_app_id  n_eval_targets             slice_name
0       0        812140               1  slice_b_single_target
1       1        485510               1  slice_b_single_target
2       2        646570               2   slice_a_multi_target

## Embed + Pool + Blend Per Candidate Embedder

In [4]:
from sentence_transformers import SentenceTransformer

BGE_QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

EMBEDDERS = {
    "all-mpnet-base-v2": {"model_name": "sentence-transformers/all-mpnet-base-v2", "query_prefix": ""},
    "bge-small-en-v1.5": {"model_name": "BAAI/bge-small-en-v1.5", "query_prefix": BGE_QUERY_PREFIX},
}


def pool_and_blend(review_df: pd.DataFrame, review_vecs: np.ndarray, description_vecs: dict[int, np.ndarray]) -> tuple[np.ndarray, np.ndarray]:
    """any_polarity__flat pooling + description blend, mirrors Stage 2's job script."""
    app_ids_out, vecs_out = [], []
    for app_id, g in review_df.groupby("app_id"):
        description_vector = description_vecs.get(int(app_id))
        if description_vector is None:
            continue
        idx = g.index.to_numpy()
        pooled = l2_normalize(review_vecs[idx].mean(axis=0))
        blended = l2_normalize((1.0 - BLEND_WEIGHT) * pooled + BLEND_WEIGHT * description_vector)
        app_ids_out.append(int(app_id))
        vecs_out.append(blended)
    return np.asarray(app_ids_out, dtype=np.int64), np.stack(vecs_out, axis=0)


catalogs: dict[str, dict] = {}
for arm_name, cfg in EMBEDDERS.items():
    print(f"--- {arm_name} ---")
    model = SentenceTransformer(cfg["model_name"], device="cuda")

    review_vecs = np.asarray(
        model.encode(review_chunks["text"].tolist(), batch_size=128, show_progress_bar=True, normalize_embeddings=False)
    )
    desc_app_ids = list(description_by_app.keys())
    desc_vecs_arr = np.asarray(
        model.encode([description_by_app[a] for a in desc_app_ids], batch_size=128, show_progress_bar=True, normalize_embeddings=False)
    )
    description_vecs = {a: v for a, v in zip(desc_app_ids, desc_vecs_arr)}

    app_ids, catalog_matrix = pool_and_blend(review_chunks, review_vecs, description_vecs)
    print(f"catalog: {catalog_matrix.shape}")

    query_texts = [cfg["query_prefix"] + t for t in examples_df["query_text"].tolist()]
    query_vecs = np.asarray(
        model.encode(query_texts, batch_size=128, show_progress_bar=True, normalize_embeddings=False)
    )
    query_vecs = np.stack([l2_normalize(v) for v in query_vecs], axis=0)

    catalogs[arm_name] = {"app_ids": app_ids, "catalog_matrix": catalog_matrix, "query_vecs": query_vecs}
    del model
    import torch
    torch.cuda.empty_cache()

print("done embedding all arms")

--- all-mpnet-base-v2 ---
catalog: (315, 768)
--- bge-small-en-v1.5 ---
catalog: (315, 384)
done embedding all arms


## Score Against Catalog, Compute Retrieval Metrics

In [5]:
def score_examples(app_ids: np.ndarray, catalog_matrix: np.ndarray, query_vecs: np.ndarray) -> pd.DataFrame:
    app_to_row = {int(a): i for i, a in enumerate(app_ids)}
    rows = []
    for i, ex in enumerate(examples_df.itertuples(index=False)):
        scores = catalog_matrix @ query_vecs[i]
        self_row = app_to_row.get(int(ex.query_app_id))
        if self_row is not None:
            scores[self_row] = -np.inf
        ranked_rows = np.argsort(-scores)[:K_RETRIEVAL]
        rows.append(
            {
                "ex_idx": ex.ex_idx,
                "slice_name": ex.slice_name,
                "Hit@K": hit_rate_at_k(ranked_rows, ex.positives, K_RETRIEVAL, app_ids),
                "Precision@K": precision_at_k(ranked_rows, ex.positives, K_RETRIEVAL, app_ids),
                "Recall@K": recall_at_k(ranked_rows, ex.positives, K_RETRIEVAL, app_ids),
            }
        )
    return pd.DataFrame(rows)


per_example_metrics: dict[str, pd.DataFrame] = {}
for arm_name, c in catalogs.items():
    per_example_metrics[arm_name] = score_examples(c["app_ids"], c["catalog_matrix"], c["query_vecs"])
    print(f"scored {arm_name}: {len(per_example_metrics[arm_name]):,} examples")

scored all-mpnet-base-v2: 12,500 examples
scored bge-small-en-v1.5: 12,500 examples


## Comparison Table

In [6]:
def summarize(df: pd.DataFrame) -> dict:
    overall = df[["Hit@K", "Precision@K", "Recall@K"]].mean()
    by_slice = df.groupby("slice_name")[["Hit@K", "Precision@K", "Recall@K"]].mean()
    return {
        "overall_hit": overall["Hit@K"],
        "slice_a_recall": by_slice.loc["slice_a_multi_target", "Recall@K"] if "slice_a_multi_target" in by_slice.index else float("nan"),
        "slice_b_hit": by_slice.loc["slice_b_single_target", "Hit@K"] if "slice_b_single_target" in by_slice.index else float("nan"),
    }

rows = []

# current (USE) -- pulled from the existing rag_v1 run's own CSVs, not recomputed.
overall_csv = pd.read_csv(RUN_DIR / "eval_retrieval_overall.csv")
by_slice_csv = pd.read_csv(RUN_DIR / "eval_retrieval_by_slice.csv")
method = "rag_chunk_v1_query_plus_desc"
current_overall_hit = overall_csv.loc[overall_csv["method"] == method, "Hit@K"].iloc[0]
current_slice_a_recall = by_slice_csv.loc[
    (by_slice_csv["method"] == method) & (by_slice_csv["slice_name"] == "slice_a_multi_target"), "Recall@K"
].iloc[0]
current_slice_b_hit = by_slice_csv.loc[
    (by_slice_csv["method"] == method) & (by_slice_csv["slice_name"] == "slice_b_single_target"), "Hit@K"
].iloc[0]
rows.append({"embedder": "current (USE)", "Hit@K (overall)": current_overall_hit, "Recall@K (Slice A)": current_slice_a_recall, "Hit@K (Slice B)": current_slice_b_hit})

for arm_name, df in per_example_metrics.items():
    s = summarize(df)
    rows.append({"embedder": arm_name, "Hit@K (overall)": s["overall_hit"], "Recall@K (Slice A)": s["slice_a_recall"], "Hit@K (Slice B)": s["slice_b_hit"]})

comparison_df = pd.DataFrame(rows).set_index("embedder").round(3)
display(comparison_df)

                   Hit@K (overall)  Recall@K (Slice A)  Hit@K (Slice B)
embedder                                                               
current (USE)                0.475               0.453            0.456
all-mpnet-base-v2            0.448               0.425            0.429
bge-small-en-v1.5            0.516               0.485            0.499

## Key Findings

| embedder | Hit@K (overall) | Recall@K (Slice A, primary) | Hit@K (Slice B, primary) |
|---|---|---|---|
| `two_tower_v1` (bar, from plan doc) | 0.512 | 0.460 | 0.496 |
| `current` (USE) | 0.475 | 0.453 | 0.456 |
| `all-mpnet-base-v2` | 0.448 | 0.425 | 0.429 |
| **`bge-small-en-v1.5`** | **0.516** | **0.485** | **0.499** |

- `bge-small-en-v1.5` clears `two_tower_v1` on all three cuts -- first RAG arm to do so.
- `all-mpnet-base-v2` regresses vs. USE -- size/generality alone didn't help; bge's
  retrieval-specific training (asymmetric query/passage, instruction prefix) looks like the real
  lever.
- Caveat: fast prototype, not contract-conformant -- in-memory flat cosine over the 315-game
  catalog, not the real Chroma/eval-job path. Mathematically equivalent (RAG only covers those
  315 games either way) but not yet run through the actual eval job/baseline machinery.

## Recommendation / Next Steps

- Promote `bge-small-en-v1.5` into a real Stage 2/3 rebuild before calling this shippable: new
  384-dim Chroma collection, new query-time embed backend, real run through
  `recs_job_eval_offline.py` to confirm this prototype number.
- Drop `all-mpnet-base-v2` -- underperforms USE.
- Once promoted, re-run the franchise/marketing qualitative check (`recs_023`) against the new
  embedder -- unknown whether the NieR:Automata-style regression persists.